In [21]:
# If needed, install deps. You can skip if already installed.
# In some managed environments these commands may be restricted; run locally if so.
# !pip install openai python-dotenv

In [22]:
from __future__ import annotations
import os, json
from dataclasses import dataclass
from typing import List, Dict, Any

# Load .env if available (safe to ignore if not present)
try:
    from dotenv import load_dotenv  # type: ignore
    load_dotenv()
except Exception:
    pass

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
USE_MOCK = not bool(OPENAI_API_KEY)

print("Mode:", "MOCK (no API key found)" if USE_MOCK else "OPENAI (API key detected)")

Mode: MOCK (no API key found)


In [23]:
class LLMClient:
    """
    A tiny wrapper:
    - If OPENAI_API_KEY is set: send real requests to OpenAI
    - Otherwise: return a clear MOCK response so you can demo without any key
    """
    def __init__(self):
        self.mode = "MOCK" if USE_MOCK else "OPENAI"
        self._client = None
        if not USE_MOCK:
            try:
                from openai import OpenAI  # type: ignore
                self._client = OpenAI(api_key=OPENAI_API_KEY)
            except Exception as e:
                print("[WARN] Falling back to MOCK mode:", e)
                self.mode = "MOCK"
                self._client = None

    def chat(self, system: str, user: str) -> str:
        if self.mode == "MOCK":
            return (
                "[MOCK REPLY]\n"
                f"System: {system.split('. ')[0]}...\n"
                f"User: {user}\n"
                "Response: This is a placeholder response. Add OPENAI_API_KEY for real outputs."
            )
        # Real API call kept intentionally simple
        resp = self._client.chat.completions.create(
            model="gpt-5-thinking",  # swap to your available model if needed
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            temperature=0.4,
        )
        return resp.choices[0].message.content or ""

llm = LLMClient()

In [24]:
@dataclass
class Persona:
    name: str
    goals: List[str]
    tone: str
    style_rules: List[str]
    tools_allowed: List[str]

    def system_prompt(self) -> str:
        return (
            f"You are the {self.name} persona.\n"
            f"Goals: {', '.join(self.goals)}.\n"
            f"Tone: {self.tone}.\n"
            f"Style rules: {', '.join(self.style_rules)}.\n"
            "Always produce clear, actionable steps."
        )

EDUCATOR = Persona(
    name="Educator",
    goals=[
        "Bring real-world projects into curriculum",
        "Map outcomes to industry tools (cloud, Git, MLOps, advanced AI)",
    ],
    tone="warm, practical, coach-like",
    style_rules=["use checklists", "include rubrics", "be concise"],
    tools_allowed=["propose_project", "make_rubric", "map_outcomes_to_tools"],
)

LEARNER = Persona(
    name="Learner",
    goals=[
        "Sharpen practical skills in cloud, Git, MLOps, advanced AI",
        "Create personal practice plans",
    ],
    tone="encouraging, direct",
    style_rules=["show commands", "weekly plan", "include checkpoints"],
    tools_allowed=["skills_gap", "study_plan", "checkpoint_quiz"],
)

EMPLOYER = Persona(
    name="Employer",
    goals=[
        "Align pipelines with training programs",
        "Define job-ready skills and scorecards",
    ],
    tone="succinct, metrics-driven",
    style_rules=["bullets", "KPIs", "templates that scale"],
    tools_allowed=["role_matrix", "assignment_brief", "screening_scorecard"],
)

PERSONAS: Dict[str, Persona] = {
    "educator": EDUCATOR,
    "learner": LEARNER,
    "employer": EMPLOYER,
}


In [25]:
# ---------- Educator tools ----------
def propose_project(topic: str, duration_weeks: int = 2) -> Dict[str, Any]:
    return {
        "title": f"{topic} — Real-World Lab",
        "duration_weeks": duration_weeks,
        "outcomes": [
            "Use Git & pull requests",
            "Deploy a minimal model/service to cloud",
            "Automate tests with CI",
        ],
        "deliverables": [
            "Repo with README and dataset notes",
            "CI workflow file",
            "Short demo video or notebook",
        ],
        "week_by_week": [
            {"week": 1, "focus": "Data + baseline; Git workflow"},
            {"week": 2, "focus": "Cloud deploy + CI; demo draft"},
        ],
    }

In [26]:
def make_rubric(outcomes: List[str]) -> Dict[str, Any]:
    return {"rubric": [{"criterion": o, "points": 10} for o in outcomes]}

In [27]:
def map_outcomes_to_tools(outcomes: List[str]) -> Dict[str, Any]:
    mapping = []
    for o in outcomes:
        tools = []
        if "git" in o.lower():
            tools += ["Git", "GitHub", "PR reviews"]
        if "cloud" in o.lower():
            tools += ["Azure", "AWS", "GCP", "Docker"]
        if "ci" in o.lower() or "automate" in o.lower():
            tools += ["GitHub Actions", "pytest", "ruff/flake8"]
        mapping.append({"outcome": o, "tools": tools or ["Choose best-fit tools"]})
    return {"mapping": mapping}


In [28]:
# ---------- Learner tools ----------
def skills_gap(current: List[str], target_role: str) -> Dict[str, Any]:
    role_basics = {
        "data_analyst": ["SQL", "Python", "Pandas", "BI dashboard", "Git"],
        "ml_engineer": ["Python", "ML basics", "Docker", "CI/CD", "Cloud deploy"],
    }
    target = role_basics.get(target_role.lower(), [])
    gaps = [s for s in target if s not in current]
    return {"target_role": target_role, "required": target, "gaps": gaps}

In [29]:
def study_plan(gaps: List[str], weeks: int = 4) -> Dict[str, Any]:
    plan: List[Dict[str, Any]] = []
    if weeks < 1:
        weeks = 1
    per_week = max(1, len(gaps) // weeks or 1)
    idx = 0
    for w in range(1, weeks + 1):
        chunk = gaps[idx: idx + per_week]
        plan.append({
            "week": w,
            "focus": chunk or ["Review & integrate"],
            "checkpoints": ["Mini-project", "1-page reflection", "Push to Git, open PR"],
        })
        idx += per_week
    return {"weeks": weeks, "plan": plan}

In [30]:
def checkpoint_quiz(topic: str) -> Dict[str, Any]:
    return {
        "topic": topic,
        "questions": [
            {"q": f"Explain {topic} in your own words.", "answer_key": "Clear and concrete."},
            {"q": f"Show a code snippet that uses {topic}.", "answer_key": "Runs and is idiomatic."},
        ],
    }

In [31]:
# ---------- Employer tools ----------
def role_matrix(roles: List[str]) -> Dict[str, Any]:
    axes = ["Core Skills", "Tools", "Day-1 Tasks", "Signals"]
    matrix = []
    for r in roles:
        matrix.append({
            "role": r,
            "Core Skills": ["Problem framing", "Data literacy", "Collaboration"],
            "Tools": ["Git", "SQL/Python", "BI or ML stack"],
            "Day-1 Tasks": ["Readme setup", "Data pull", "Small change & PR"],
            "Signals": ["Portfolio repos", "PR history", "Take-home quality"],
        })
    return {"axes": axes, "matrix": matrix}

In [32]:
def assignment_brief(role: str) -> Dict[str, Any]:
    return {
        "role": role,
        "duration": "48–72h",
        "context": "Mirror a small week-one task.",
        "requirements": [
            "Fork starter repo; open PRs",
            "Implement feature/analysis; write tests",
            "Short Loom video walkthrough",
        ],
        "evaluation": ["Correctness", "Code quality & tests", "Docs & comms"],
    }

In [33]:
def screening_scorecard(role: str) -> Dict[str, Any]:
    return {
        "role": role,
        "dimensions": [
            {"name": "Technical Baseline", "weight": 0.4},
            {"name": "Execution/Ownership", "weight": 0.3},
            {"name": "Communication", "weight": 0.2},
            {"name": "Team Fit", "weight": 0.1},
        ],
        "scales": {"0": "Not demonstrated", "1": "Basic", "2": "Good", "3": "Excellent"},
    }

In [34]:
TOOL_REGISTRY = {
    # Educator
    "propose_project": propose_project,
    "make_rubric": make_rubric,
    "map_outcomes_to_tools": map_outcomes_to_tools,
    # Learner
    "skills_gap": skills_gap,
    "study_plan": study_plan,
    "checkpoint_quiz": checkpoint_quiz,
    # Employer
    "role_matrix": role_matrix,
    "assignment_brief": assignment_brief,
    "screening_scorecard": screening_scorecard,
}

In [35]:
def use_tool(tool_name: str, **kwargs) -> Dict[str, Any]:
    """
    Call a built-in tool by name:
      use_tool("propose_project", topic="Model Monitoring", duration_weeks=2)
    """
    if tool_name not in TOOL_REGISTRY:
        return {"error": f"Unknown tool: {tool_name}"}
    try:
        return TOOL_REGISTRY[tool_name](**kwargs)
    except TypeError as e:
        return {"error": f"Bad arguments for {tool_name}: {e}"}

In [36]:
def ask(persona_key: str, message: str) -> str:
    """
    Send a free-form question to the AI (or MOCK if no key).
      ask("learner", "Give me a 2-week plan for Docker + CI basics.")
    """
    key = persona_key.strip().lower()
    if key not in PERSONAS:
        raise ValueError(f"Unknown persona '{persona_key}'. Try one of {list(PERSONAS.keys())}")
    system = PERSONAS[key].system_prompt()
    return llm.chat(system, message)

In [37]:
# -- Educator: project brief
educator_project = use_tool("propose_project", topic="Model Monitoring", duration_weeks=2)
educator_project


{'title': 'Model Monitoring — Real-World Lab',
 'duration_weeks': 2,
 'outcomes': ['Use Git & pull requests',
  'Deploy a minimal model/service to cloud',
  'Automate tests with CI'],
 'deliverables': ['Repo with README and dataset notes',
  'CI workflow file',
  'Short demo video or notebook'],
 'week_by_week': [{'week': 1, 'focus': 'Data + baseline; Git workflow'},
  {'week': 2, 'focus': 'Cloud deploy + CI; demo draft'}]}

In [38]:
# -- Learner: gap -> plan
gap = use_tool("skills_gap", current=["Python", "Git"], target_role="data_analyst")
plan = use_tool("study_plan", gaps=gap["gaps"], weeks=4)
gap, plan

({'target_role': 'data_analyst',
  'required': ['SQL', 'Python', 'Pandas', 'BI dashboard', 'Git'],
  'gaps': ['SQL', 'Pandas', 'BI dashboard']},
 {'weeks': 4,
  'plan': [{'week': 1,
    'focus': ['SQL'],
    'checkpoints': ['Mini-project',
     '1-page reflection',
     'Push to Git, open PR']},
   {'week': 2,
    'focus': ['Pandas'],
    'checkpoints': ['Mini-project',
     '1-page reflection',
     'Push to Git, open PR']},
   {'week': 3,
    'focus': ['BI dashboard'],
    'checkpoints': ['Mini-project',
     '1-page reflection',
     'Push to Git, open PR']},
   {'week': 4,
    'focus': ['Review & integrate'],
    'checkpoints': ['Mini-project',
     '1-page reflection',
     'Push to Git, open PR']}]})

In [39]:
# -- Employer: role matrix
matrix = use_tool("role_matrix", roles=["Data Analyst", "ML Ops Intern"])
matrix

{'axes': ['Core Skills', 'Tools', 'Day-1 Tasks', 'Signals'],
 'matrix': [{'role': 'Data Analyst',
   'Core Skills': ['Problem framing', 'Data literacy', 'Collaboration'],
   'Tools': ['Git', 'SQL/Python', 'BI or ML stack'],
   'Day-1 Tasks': ['Readme setup', 'Data pull', 'Small change & PR'],
   'Signals': ['Portfolio repos', 'PR history', 'Take-home quality']},
  {'role': 'ML Ops Intern',
   'Core Skills': ['Problem framing', 'Data literacy', 'Collaboration'],
   'Tools': ['Git', 'SQL/Python', 'BI or ML stack'],
   'Day-1 Tasks': ['Readme setup', 'Data pull', 'Small change & PR'],
   'Signals': ['Portfolio repos', 'PR history', 'Take-home quality']}]}

In [40]:
# --- Force MOCK mode for this session ---
USE_MOCK = True
try:
    llm.mode = "MOCK"
    llm._client = None
except NameError:
    pass  # if llm isn't defined yet, it's fine
print("Mode:", "MOCK")


Mode: MOCK


In [41]:
# -- Ask the AI (will return MOCK text if no API key is set)
ask("educator", "Create a 2-week lab outline for GitHub Actions + Azure.")

'[MOCK REPLY]\nSystem: You are the Educator persona.\nGoals: Bring real-world projects into curriculum, Map outcomes to industry tools (cloud, Git, MLOps, advanced AI).\nTone: warm, practical, coach-like.\nStyle rules: use checklists, include rubrics, be concise.\nAlways produce clear, actionable steps....\nUser: Create a 2-week lab outline for GitHub Actions + Azure.\nResponse: This is a placeholder response. Add OPENAI_API_KEY for real outputs.'

In [ ]:
pip install streamlit


Note: you may need to restart the kernel to use updated packages.


: 